# Hybrid Woodelf Experiment

Compares 4 Woodelf implementations across tree depths on the Fraud Detection dataset:

| Approach | `use_sparse_approaches` | `use_faster_mn_p2s` |
|---|---|---|
| `HybridWoodelf (Sparse)` | True | True |
| `HybridWoodelf (Sparse, slow MN)` | True | False |
| `HybridWoodelf (Auto)` | False | True |
| `WoodelfHD` (baseline) | — | — |

**Task types:** Path-Dependent SHAP, Background SHAP (m=100k), Path-Dependent Interactions.

### Flow
1. Mount Drive → configure paths
2. Clone repos & install
3. (Optional) Restore partial results from Drive for resume
4. **Launch all 4 approaches in parallel** — each runs as a background subprocess
5. Monitor progress (re-run anytime)
6. Wait for all processes to finish
7. *(Run separately when all done)* Generate HTML report → save to Drive

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
import pathlib

DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/treebranchmarks/hybrid_woodelf_sweep')
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

# One Drive result file per method — written incrementally during the run.
METHODS = {
    'woodelf_hybrid_sparse':            DRIVE_FOLDER / 'hybrid_sparse.json',
    'woodelf_hybrid_sparse_no_fast_mn': DRIVE_FOLDER / 'hybrid_sparse_no_fast_mn.json',
    'woodelf_hybrid_auto':              DRIVE_FOLDER / 'hybrid_auto.json',
    'woodelf_hd':                       DRIVE_FOLDER / 'woodelf_hd.json',
}

EXPERIMENT_NAME = 'fraud_hybrid_woodelf_experiment'
EXPERIMENT_MODULE = 'benchmarks.wip.fraud_hybrid_woodelf_experiment'

# Both paths are needed in PYTHONPATH for subprocesses:
#   /content/treebranchmarks  — makes `benchmarks` importable via -m
#   /content/woodelf_explainer — makes `import woodelf` resolve correctly
#     (the dist name is woodelf_explainer, not woodelf)
PYTHONPATH = '/content/treebranchmarks:/content/woodelf_explainer'

print(f'Drive folder: {DRIVE_FOLDER}')
for method, path in METHODS.items():
    status = '(exists)' if path.exists() else '(new)'
    print(f'  {method}: {path.name} {status}')

In [ ]:
# ── Step 3: Clone repositories ───────────────────────────────────────────────
TREEBRANCHMARKS_URL = 'https://github.com/ron-wettenstein/TreeBranchMarks.git'

# woodelf: clone the feature branch directly
!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks
!git clone -b feature/hybrid_background_shap \
    https://github.com/ron-wettenstein/woodelf.git \
    /content/woodelf_explainer

# Confirm woodelf branch
!git -C /content/woodelf_explainer branch --show-current

In [ ]:
# ── Step 4: Install packages ─────────────────────────────────────────────────
# pip install handles transitive deps (numpy, pandas, shap, …).
# sys.path inserts make imports work correctly in this kernel:
#   woodelf_explainer → `import woodelf` (dist name mismatch)
#   treebranchmarks   → `import benchmarks` (needed for the report cell)
import sys

!pip install -q -e /content/woodelf_explainer
!pip install -q -e /content/treebranchmarks

for path in ['/content/woodelf_explainer', '/content/treebranchmarks']:
    if path not in sys.path:
        sys.path.insert(0, path)

# Verify
import woodelf
from woodelf.hybrid_woodelf import hybrid_woodelf
print(f'woodelf imported OK (version: {woodelf.__version__})')
print('hybrid_woodelf OK')

In [ ]:
# ── Step 5: Restore method caches from Drive (resume after interruption) ──────
import shutil, pathlib

cache_dir = pathlib.Path(f'/content/treebranchmarks/cache/method_results/{EXPERIMENT_NAME}')
cache_dir.mkdir(parents=True, exist_ok=True)

for method, drive_path in METHODS.items():
    local = cache_dir / f'{method}.json'
    if drive_path.exists() and not local.exists():
        shutil.copy(drive_path, local)
        print(f'Restored {method}.json ({drive_path.stat().st_size // 1024} KB)')
    elif local.exists():
        print(f'Cache already present: {method}.json')
    else:
        print(f'No existing cache for {method} — starting fresh')

In [ ]:
# ── Step 6: Launch all 4 approaches in parallel ───────────────────────────────
# Each approach runs as an independent subprocess writing to its own log file.
# This cell returns immediately; use the monitor cell below to check progress.
import subprocess, sys, os, pathlib

pathlib.Path('/content/logs').mkdir(exist_ok=True)

env = os.environ.copy()
env['PYTHONPATH'] = PYTHONPATH + ':' + env.get('PYTHONPATH', '')

procs = {}
for method, drive_path in METHODS.items():
    log = open(f'/content/logs/{method}.log', 'w')
    p = subprocess.Popen(
        [
            sys.executable, '-u', '-m', EXPERIMENT_MODULE,
            '--method', method,
            '--result_location', str(drive_path),
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        env=env,
    )
    procs[method] = (p, log)
    print(f'▶ Started {method}  (PID {p.pid})')

print('\nAll 4 processes launched. Run the monitor cell to check progress.')

In [ ]:
# ── Monitor: re-run this cell anytime to check progress ──────────────────────
import pathlib

for method, (p, _) in procs.items():
    rc = p.poll()  # None = still running
    status = 'running' if rc is None else (f'✓ done (rc={rc})' if rc == 0 else f'✗ FAILED (rc={rc})')
    log_path = pathlib.Path(f'/content/logs/{method}.log')
    lines = log_path.read_text().splitlines() if log_path.exists() else []
    tail = lines[-5:] if lines else ['(no output yet)']
    print(f'\n── {method}  [{status}] ──')
    for line in tail:
        print(f'  {line}')

In [ ]:
# ── Wait for all processes to finish ─────────────────────────────────────────
# Run this cell when you are ready to block until everything completes.
print('Waiting for all processes...')
for method, (p, log) in procs.items():
    rc = p.wait()
    log.close()
    status = '✓ OK' if rc == 0 else f'✗ FAILED (rc={rc})'
    print(f'  {method}: {status}')

print('\nAll done. Run the HTML report cell when ready.')

In [ ]:
# ── Generate HTML report ──────────────────────────────────────────────────────
# Run this cell only after all 4 processes have finished.
# It copies the Drive result files into the local cache, runs the experiment
# (all cache hits — no recomputation), and generates the HTML report.
import shutil, os, pathlib, sys

for path in ['/content/woodelf_explainer', '/content/treebranchmarks']:
    if path not in sys.path:
        sys.path.insert(0, path)

# Copy Drive results → local method cache
cache_dir = pathlib.Path(f'/content/treebranchmarks/cache/method_results/{EXPERIMENT_NAME}')
cache_dir.mkdir(parents=True, exist_ok=True)

for method, drive_path in METHODS.items():
    if not drive_path.exists():
        raise FileNotFoundError(f'Missing Drive result for {method}: {drive_path}')
    shutil.copy(drive_path, cache_dir / f'{method}.json')
    print(f'Copied {method}.json ({drive_path.stat().st_size // 1024} KB)')

# Run experiment (all results served from cache) and generate report
from benchmarks.wip.fraud_hybrid_woodelf_experiment import build_experiment

exp = build_experiment()
exp.run()
html_path = exp.generate_html()

# Save HTML to Drive
REPORT_HTML = DRIVE_FOLDER / f'{EXPERIMENT_NAME}.html'
shutil.copy(html_path, REPORT_HTML)
print(f'\nHTML report saved to Drive: {REPORT_HTML}')

# Download to local machine
from google.colab import files
files.download(str(REPORT_HTML))